# R2: Harmonise and merge

**Goal:** turn the four raw files into one table with the same columns for every year.
Every change is counted, so it can be traced back. Raw files stay untouched.

**Rule:** never print rows of the raw data. They contain farmer names and phone numbers.

**Outputs go to** `results/phase1a/` (tables) and `data/processed_v2/` (data).

## Step 1: Mark the two survey batches

2022 to 2024 each hold two survey batches merged together. The `layer` column tells them apart:
"MH-Latur-..." is batch B, the other value is batch A. 2025 is a single file.
As a cross-check, batch A should fill `Area_acres` and batch B should fill `PolyArea`, never both.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import geopandas as gpd

here = Path.cwd()
ROOT = here if (here / "data" / "raw").exists() else here.parent
RAW = ROOT / "data" / "raw"
OUT = ROOT / "results" / "phase1a"
OUT.mkdir(parents=True, exist_ok=True)
DATA2 = ROOT / "data" / "processed_v2"
DATA2.mkdir(parents=True, exist_ok=True)

YEARS = [2022, 2023, 2024, 2025]
raw = {y: gpd.read_file(RAW / f"Latur_GT_ND_{y}.shp") for y in YEARS}

def farmer_code(g):
    """Anonymous code per farmer (name + phone), same as in R1."""
    key = (g["Farmer"].astype(str).str.strip().str.lower() + "|" +
           g["Phone"].astype(str).str.strip())
    return key.apply(lambda s: hashlib.sha1(s.encode()).hexdigest()[:10])

rows = []
for y, g in raw.items():
    g["year"] = y
    if "layer" in g:
        # batch B came from the "MH-Latur-..." export; batch A is the other one
        g["batch"] = np.where(g["layer"].astype(str).str.startswith("MH-Latur"), "B", "A")
    else:
        g["batch"] = "2025"
    g["farmer_code"] = farmer_code(g)

    # cross-check: batch A should fill Area_acres, batch B should fill PolyArea
    if "PolyArea" in g:
        a_ok = (g.loc[g["batch"] == "A", "Area_acres"].notna()).all() & (g.loc[g["batch"] == "A", "PolyArea"].isna()).all()
        b_ok = (g.loc[g["batch"] == "B", "PolyArea"].notna()).all() & (g.loc[g["batch"] == "B", "Area_acres"].isna()).all()
    else:
        a_ok = b_ok = True
    counts = g["batch"].value_counts().reindex(["A", "B", "2025"], fill_value=0)
    rows.append({"year": y, **counts.to_dict(), "matches_area_columns": bool(a_ok and b_ok)})

batches = pd.DataFrame(rows)
batches.to_csv(OUT / "R2_batches.csv", index=False)
print(batches.to_string(index=False))
print("total fields:", sum(len(g) for g in raw.values()))

 year  A   B  2025  matches_area_columns
 2022 22  33     0                  True
 2023 12  48     0                  True
 2024 23 300     0                  True
 2025  0   0   769                  True
total fields: 1207
